# 数据导入：10 万行行为数据入库 MySQL

完整 ETL 链路：CSV → pandas 清洗 → MySQL → 读回验证。

> 运行前：右上角选择内核（Select Kernel）→ Python 3.12

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, URL

## 1. 读取原始数据

数据字段：`user_id` 用户ID、`item_id` 商品ID、`category_id` 品类ID、`behavior_type` 行为类型（pv浏览/fav收藏/cart加购/buy购买）、`timestamp` 行为发生时间戳

In [2]:
# 读取原始行为数据（10 万行）
df = pd.read_csv(
    r"C:\Users\wukeyu31434\data-analyst-projects\taobao-user-behavior-analysis\data\raw_user_behavior_sample.csv"
)
print(f"数据规模: {df.shape[0]:,} 行 × {df.shape[1]} 列")
df.head()

数据规模: 100,000 行 × 5 列


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


## 2. 数据质量检查

看结构 → 查空值 → 查重复

In [3]:
# 2.1 看结构：每列类型、非空数量、内存占用
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   user_id        100000 non-null  int64
 1   item_id        100000 non-null  int64
 2   category_id    100000 non-null  int64
 3   behavior_type  100000 non-null  str  
 4   timestamp      100000 non-null  int64
dtypes: int64(4), str(1)
memory usage: 3.8 MB


In [4]:
# 2.2 查空值：每一列有多少个空
print("各列空值数量:")
print(df.isnull().sum())

各列空值数量:
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64


In [5]:
# 2.3 查重复：完全重复的行有多少
print(f"完全重复的行数: {df.duplicated().sum():,}")
print(f"时间戳范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")

完全重复的行数: 0
时间戳范围: 1505117799 ~ 1512316763


## 3. 基础清洗

1. 重命名 `timestamp` → `ts`，与数据库表列名对齐（`TIMESTAMP` 是 MySQL 保留关键字）
2. 去除完全重复的行

In [6]:
# 3.1 重命名列
df = df.rename(columns={"timestamp": "ts"})
print("列名:", df.columns.tolist())

列名: ['user_id', 'item_id', 'category_id', 'behavior_type', 'ts']


In [7]:
# 3.2 去重
before = len(df)
df = df.drop_duplicates()
print(f"去重: {before:,} → {len(df):,} 行")

去重: 100,000 → 100,000 行


## 4. 写入 MySQL

密码从环境变量 `MYSQL_PASSWORD` 读取（`setx MYSQL_PASSWORD "你的密码"` 后重启 VS Code 生效），代码中不存密码。`chunksize=10000` 分批写入，控制内存占用。

In [8]:
url = URL.create(
    "mysql+pymysql",
    username="root",
    password=os.environ["MYSQL_PASSWORD"],
    host="localhost",
    port=3306,
    database="taobao_analysis",
    query={"charset": "utf8mb4"},
)
engine = create_engine(url)
print("连接成功")

连接成功


In [9]:
# 写入（if_exists='replace' = 表已存在则重建，重复执行不受影响）
df.to_sql(
    name="user_behavior",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
)
print("导入完成")

导入完成


## 5. 验证：从数据库读回

导入后必须校验行数与内容。

In [10]:
# 5.1 行数应与 df 一致
total = pd.read_sql("SELECT COUNT(*) AS total FROM user_behavior", con=engine)
print(f"数据库行数: {total['total'].iloc[0]:,}  |  预期: {len(df):,}")
assert total['total'].iloc[0] == len(df), "行数不一致！"
print("行数一致")

数据库行数: 100,000  |  预期: 100,000
行数一致


In [11]:
# 5.2 抽样核对内容
pd.read_sql("SELECT * FROM user_behavior LIMIT 5", con=engine)

,user_id,item_id,category_id,behavior_type,ts
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


## 6. 业务查询示例

用户每种行为各发生多少次？

In [12]:
pd.read_sql(
    """
    SELECT behavior_type, COUNT(*) AS cnt
    FROM user_behavior
    GROUP BY behavior_type
    ORDER BY cnt DESC
    """,
    con=engine,
)

,behavior_type,cnt
0,pv,89709
1,cart,5446
2,fav,2744
3,buy,2101
